# Surface confidence assessment & stratigraphic boundaries and faults overlap for 3D geological models

Runs the horizontal/vertical/combined confidence pipeline, the symmetric (bidirectional)
boundary-overlap metric, and the fault validation/throw analyses on this custom
dataset using the
**original (non-resampled)** topo-intersection and geological-map-boundary shapefiles
(`3D_Topo_Intersections.shp`, `Limiti_stratigrafici_carta_geologica_DEF.shp`,
`3D_Topo_Intersections_Faults.shp`, `Faglie_carta_geologica_DEF.shp`). All layers are
standardized to a single CRS, EPSG:6707.

See [`CONTEXT.md`](./CONTEXT.md) for how this dataset maps onto [`THEORY.md`](./THEORY.md), and [`run_accuracy_original.py`](./run_accuracy_original.py) for the equivalent standalone script (kept in sync with this notebook).

Outputs are written to `./output_results_original/`.

## 0. Load workspace (Colab)

If you are running this notebook on **Google Colab**, run the cell below first - it
clones this repository (data included) and installs the requirements. If you are
running locally with `requirements.txt` already installed, you can skip or run it
anyway (it is a no-op outside Colab).

In [1]:
# ### LOAD WORKSPACE
# Run this cell first. On Google Colab it clones this repository (which already
# contains working_files_folder/ with all the input data) and installs requirements.
# When running locally, it just confirms you're in the right folder.
import os

REPO_URL = "https://github.com/DanieleBuson/GeoSurface_Accuracy_custom.git"
REPO_DIR = "/content/GeoSurface_Accuracy_custom"

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not os.path.exists(REPO_DIR):
        !git clone {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}
    %pip install -q -r requirements.txt
else:
    print(f"Not running on Colab - using current working directory: {os.getcwd()}")


Not running on Colab - using current working directory: /Users/danielebuson/Documents/projects/to-remove/GeoSurface_Accuracy_custom


## 1. Imports

In [2]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd

from custom_utils import (
    read_gocad_ts_multi,
    generate_accuracy_outputs,
    generate_vertical_outputs,
    generate_combined_confidence,
    visualize_data,
    standardize_crs,
    clean_formation_name,
)
from custom_validation import (
    STRAT_MAP,
    select_map_lines_strat,
    generate_enhanced_boundary_overlap,
    compute_fault_throw_per_horizon,
    generate_fault_validation_outputs,
    compute_unit_thickness_at_grid,
    generate_acceptance_table,
)

## 2. Configuration

In [3]:
# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
WORKING_DIR = "working_files_folder"
OUTPUT_DIR = "output_results_original"

TS_HORIZONS_FILE = os.path.join("original", "GOCAD_ASCII_All.ts")
TS_FAULTS_FILE = os.path.join("original", "GOCAD_ASCII_Faults.ts")

SECTIONS_FILE = "3D_SectionsGrid.shp"
MAPS_FILE = "Limiti_stratigrafici_carta_geologica_DEF.shp"  # sole GIS source for stratigraphic contacts
TOPO_FILE = "3D_Topo_Intersections.shp"          # horizon topo-intersections
FAULTS_TOPO_FILE = "3D_Topo_Intersections_Faults.shp"
FAGLIE_FILE = "Faglie_carta_geologica_DEF.shp"   # sole GIS source for faults

CRS_MODEL = "EPSG:6707"   # RDN2008/UTM32N — single CRS baseline for the whole pipeline

GRID_SPACING = 200.0
LINE_STEP = 200.0
MAPS_STEP = 100.0
BUFFER_DIST_M = 20.0

## 3. Pipeline (`main`)

In [4]:
def main():
    print("Starting custom geological accuracy analysis (original dataset)...")

    if not os.path.exists(WORKING_DIR):
        print(f"The folder {WORKING_DIR} does not exist.")
        return None

    # --- Load horizon surfaces ---
    ts_horiz_path = os.path.join(WORKING_DIR, TS_HORIZONS_FILE)
    surfaces_data = read_gocad_ts_multi(ts_horiz_path, read_thickness=True)
    if not surfaces_data:
        print("No surfaces found in the horizons .ts file.")
        return None

    # original/GOCAD_ASCII_All.ts may contain both horizons and faults; separate by prefix
    horizon_surfaces = {k: v for k, v in surfaces_data.items() if k.upper().startswith('TOP_')}
    fault_surfaces_from_all = {k: v for k, v in surfaces_data.items() if not k.upper().startswith('TOP_')}

    # Load dedicated faults file if faults not already in the All.ts
    ts_faults_path = os.path.join(WORKING_DIR, TS_FAULTS_FILE)
    if os.path.exists(ts_faults_path):
        faults_data = read_gocad_ts_multi(ts_faults_path)
        fault_surfaces = {**fault_surfaces_from_all, **faults_data}
    else:
        fault_surfaces = fault_surfaces_from_all

    print(f"Horizon surfaces ({len(horizon_surfaces)}): {list(horizon_surfaces.keys())}")
    print(f"Fault surfaces  ({len(fault_surfaces)}): {list(fault_surfaces.keys())}")

    # --- Load vector data, standardized to EPSG:6707 ---
    sections_all = standardize_crs(gpd.read_file(os.path.join(WORKING_DIR, SECTIONS_FILE)))
    maps_all = standardize_crs(gpd.read_file(os.path.join(WORKING_DIR, MAPS_FILE)))
    topo_all = standardize_crs(gpd.read_file(os.path.join(WORKING_DIR, TOPO_FILE)))

    faults_topo = standardize_crs(gpd.read_file(os.path.join(WORKING_DIR, FAULTS_TOPO_FILE)))
    faglie_gdf = standardize_crs(gpd.read_file(os.path.join(WORKING_DIR, FAGLIE_FILE)))

    print(f"Sections: {len(sections_all)}, Maps: {len(maps_all)}, "
          f"Topo: {len(topo_all)}, FaultTopo: {len(faults_topo)}, "
          f"Faglie: {len(faglie_gdf)}")

    # --- Global extents ---
    all_vert_list = [v for v in horizon_surfaces.values()
                     if v.get('vertices') is not None and len(v['vertices']) > 0]
    all_xyz = np.vstack([v['vertices'] for v in all_vert_list])
    global_xmin, global_ymin = np.min(all_xyz[:, :2], axis=0)
    global_xmax, global_ymax = np.max(all_xyz[:, :2], axis=0)

    for gdf in (sections_all, maps_all, topo_all):
        if gdf is not None and not gdf.empty:
            b = gdf.geometry.bounds
            global_xmin = min(global_xmin, b.minx.min())
            global_xmax = max(global_xmax, b.maxx.max())
            global_ymin = min(global_ymin, b.miny.min())
            global_ymax = max(global_ymax, b.maxy.max())

    global_xlim = (global_xmin, global_xmax)
    global_ylim = (global_ymin, global_ymax)

    # --- Per-horizon loop ---
    results = {}
    all_vertices = []
    enhanced_overlap_results = []

    for raw_sname, data in horizon_surfaces.items():
        # Output filenames use a cleaned formation name (modeling-software processing
        # tags like merged/resampled/split stripped) plus a single "_original" suffix
        # marking this as the complete, non-split surface this pipeline always uses.
        formation_name = clean_formation_name(raw_sname)
        sname = f"{formation_name}_original"
        print(f"\n--- Surface: {sname} ---")
        vertices = data.get('vertices')
        triangles = data.get('triangles')
        thickness = data.get('thickness')
        if vertices is None or len(vertices) == 0:
            print("  No vertices, skipping.")
            continue
        all_vertices.append(vertices)

        maps_for_surface = select_map_lines_strat(maps_all, sname)
        print(f"  Map contacts matched (strat): {len(maps_for_surface)} features")

        acc_outputs = generate_accuracy_outputs(
            vertices, None, sections_all, OUTPUT_DIR,
            use_wells=False, use_sections=True,
            use_maps=True, maps_shp=maps_all, maps_step=MAPS_STEP,
            grid_spacing=GRID_SPACING, line_step=LINE_STEP, surface_name=sname,
            xlim=global_xlim, ylim=global_ylim, crs_proj=CRS_MODEL,
            maps_lines_prefiltered=maps_for_surface,
        )

        vert_outputs = None
        try:
            # vertical_confidence_grid_<formation_name> uses the bare formation name
            # (no "_original" suffix) per the CSV/HTML/PNG naming convention.
            vert_outputs = generate_vertical_outputs(
                vertices, triangles, None, sections_all,
                acc_outputs.get('grid_points'), acc_outputs.get('GX'), acc_outputs.get('GY'),
                acc_outputs.get('mask'), OUTPUT_DIR, formation_name, idw_power=2,
                topo_shp=topo_all, xlim=global_xlim, ylim=global_ylim, crs_proj=CRS_MODEL
            )
            if vert_outputs:
                print(f"  Vertical: {vert_outputs.get('samples', 0)} checkpoints "
                      f"({vert_outputs.get('samples_topo', 0)} from topo).")
        except Exception as e:
            print(f"  Error in vertical confidence: {e}")

        combined_outputs = None
        if vert_outputs is not None:
            combined_outputs = generate_combined_confidence(
                acc_outputs, vert_outputs, OUTPUT_DIR, sname,
                crs_proj=CRS_MODEL, alpha=0.5, mode="min"
            )

        overlap_result = None
        try:
            overlap_result = generate_enhanced_boundary_overlap(
                topo_all, maps_for_surface, sname, OUTPUT_DIR,
                buffer_dist=BUFFER_DIST_M, xlim=global_xlim, ylim=global_ylim,
            )
            if overlap_result:
                print(f"  Boundary overlap: A->B={overlap_result['overlap_pct_A_to_B']:.1f}% "
                      f"B->A={overlap_result['overlap_pct_B_to_A']:.1f}% "
                      f"mean={overlap_result['overlap_pct_mean']:.1f}%")
                enhanced_overlap_results.append(overlap_result)
        except Exception as e:
            print(f"  Error in enhanced boundary overlap: {e}")
            import traceback; traceback.print_exc()

        fault_throw_out = None
        try:
            gp = acc_outputs.get('grid_points')
            GX = acc_outputs.get('GX')
            GY = acc_outputs.get('GY')
            mask = acc_outputs.get('mask')
            if gp is not None and fault_surfaces:
                fault_throw_out = compute_fault_throw_per_horizon(
                    vertices, fault_surfaces, OUTPUT_DIR, sname,
                    gp, GX, GY, mask,
                    offset_m=100, crs_proj=CRS_MODEL,
                    xlim=global_xlim, ylim=global_ylim,
                )
                if fault_throw_out:
                    print(f"  Fault throw impact: {len(fault_throw_out['throw_sample_values'])} "
                          f"sample points, mean={np.nanmean(fault_throw_out['throw_sample_values']):.1f} m")
        except Exception as e:
            print(f"  Error in fault throw per horizon: {e}")
            import traceback; traceback.print_exc()

        # acc_outputs['grid_points'] is already the masked subset
        gp_masked = acc_outputs.get('grid_points')
        thickness_at_grid = None
        if gp_masked is not None:
            try:
                thickness_at_grid = compute_unit_thickness_at_grid(
                    vertices, thickness, gp_masked
                )
                valid_t = thickness_at_grid[~np.isnan(thickness_at_grid)] if thickness_at_grid is not None and len(thickness_at_grid) > 0 else []
                if len(valid_t) > 0:
                    print(f"  Unit thickness at grid: mean={np.nanmean(valid_t):.1f} m")
                else:
                    print("  Unit thickness: not available (Thickness property absent in original .ts files)")
            except Exception as e:
                print(f"  Error in thickness interpolation: {e}")

        try:
            visualize_data(
                vertices, triangles, None, sections_all, apply_smoothing=False,
                smoothing_iterations=3, smoothing_factor=0.2, crs=CRS_MODEL,
                output_filename=f'model_dataset_{sname}.png',
                grid_points=acc_outputs.get('grid_points'), surface_name=sname,
                show_plot=False, xlim=global_xlim, ylim=global_ylim, output_dir=OUTPUT_DIR
            )
        except Exception as e:
            print(f"  Visualization error: {e}")

        results[sname] = {
            'vertices': vertices,
            'triangles': triangles,
            'grid_points': acc_outputs.get('grid_points'),
            'horizontal_weights': acc_outputs.get('weights'),
            'vertical_confidence': vert_outputs,
            'combined_confidence': combined_outputs,
            'overlap_result': overlap_result,
            'thickness_at_grid': thickness_at_grid,
            'vert_outputs': vert_outputs,
        }

    # --- Combined model footprint ---
    if all_vertices:
        try:
            visualize_data(
                np.vstack(all_vertices), None, None, sections_all, apply_smoothing=False,
                smoothing_iterations=0, smoothing_factor=0.0, crs=CRS_MODEL,
                output_filename='model_dataset.png', grid_points=None,
                surface_name='model', show_plot=False,
                xlim=global_xlim, ylim=global_ylim, output_dir=OUTPUT_DIR
            )
        except Exception as e:
            print(f"Combined visualization error: {e}")

    # --- Boundary overlap summary ---
    if enhanced_overlap_results:
        summary_path = os.path.join(OUTPUT_DIR, 'boundary_overlap_summary.csv')
        pd.DataFrame(enhanced_overlap_results).to_csv(summary_path, index=False)
        print(f"\nWrote {summary_path} ({len(enhanced_overlap_results)} rows)")

    # --- Fault validation ---
    print("\n--- Fault validation ---")
    try:
        fault_val_result = generate_fault_validation_outputs(
            faults_topo, faglie_gdf, OUTPUT_DIR, buffer_dist=BUFFER_DIST_M
        )
        if fault_val_result:
            agg = fault_val_result['aggregate'].iloc[0]
            print(f"  Fault overlap: mean={agg['mean_fault_overlap_mean']:.1f}% "
                  f"over {int(agg['n_faults_compared'])} faults (buffer {agg['buffer_dist_m']:.0f} m)")
    except Exception as e:
        print(f"Error in fault validation: {e}")
        import traceback; traceback.print_exc()

    # --- Acceptance table ---
    print("\n--- Acceptance table ---")
    try:
        generate_acceptance_table(results, OUTPUT_DIR)
    except Exception as e:
        print(f"Error in acceptance table: {e}")
        import traceback; traceback.print_exc()

    print("\nAnalysis completed (original).")
    return results

## 4. Run

In [5]:
data = main()

Starting custom geological accuracy analysis (original dataset)...


Horizon surfaces (7): ['TOP_ARV_merged_resampledoriginal', 'TOP_DPR_merged_resampledoriginal', 'TOP_LOP_merged_resampled_original', 'TOP_RTZinf_merged_resampled_original', 'TOP_FMZ_merged_resampled_original', 'TOP_RTZsup_merged_resampledoriginal', 'TOP_OSV_merged_resampledoriginal']
Fault surfaces  (12): ['F10_org', 'F3_org', 'F6_org', 'F9a_org', 'F2_org', 'F4_org', 'F7_org', 'F8b_org', 'F9b_org', 'F1_org', 'F5_org', 'F8a_org']
Sections: 22, Maps: 456, Topo: 7, FaultTopo: 12, Faglie: 232

--- Surface: TOP_ARV_original ---
  Map contacts matched (strat): 16 features


  Vertical: 3572 checkpoints (3572 from topo).


  Boundary overlap: A->B=52.5% B->A=81.2% mean=66.9%


  Fault throw impact: 6312 sample points, mean=22.0 m
  Unit thickness at grid: mean=24.8 m
Surface footprint shown as rectangle (fast mode)


...

  Boundary overlap: A->B=56.1% B->A=84.9% mean=70.5%


  Fault throw impact: 6312 sample points, mean=22.7 m
  Unit thickness at grid: mean=39.8 m
Surface footprint shown as rectangle (fast mode)
Execution time: 0.03 seconds


Figure saved to: output_results_original/model_dataset_TOP_OSV_original.png
Surface footprint shown as rectangle (fast mode)
Execution time: 0.03 seconds


Figure saved to: output_results_original/model_dataset.png

Wrote output_results_original/boundary_overlap_summary.csv (7 rows)

--- Fault validation ---
  Faults present in GIS but not in MOVE (skipped): ['17', '19']
  Wrote output_results_original/fault_validation_per_fault.csv
  Wrote output_results_original/fault_validation_aggregate.csv


  Wrote output_results_original/fault_validation_map.png
  Fault overlap: mean=76.9% over 12 faults (buffer 20 m)

--- Acceptance table ---
  Wrote output_results_original/acceptance_table.csv
  Wrote output_results_original/validation_summary.csv

Analysis completed (original).
